In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2




r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\cristiano.nascimento\AppData\Local\Temp\ipykernel_23520\847897453.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\cristiano.nascimento\git\LSTM_giroscopio_acelerometro\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
## get dataset
pd.set_option("display.precision", 9)

# select only sensor data from smart watches
data = pd.read_csv(WORKDIR+'/Code/DataPrep/activity_recognition_data_watch_concat_by_comparacao_grupos.csv',dtype={"value": "float64"})

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
#remove the null class
data=data.dropna()

# **Data Exploration**

In [ ]:
## the distribution of output classes
plt.bar(data["atividade"].value_counts().index, data["atividade"].value_counts().values)
plt.show()

In [ ]:
## x,y,z vaues for activity 'sit' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['acelerômetro_x','acelerômetro_Y','acelerômetro_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## x,y,z vaues for activity 'stand' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['giroscópio_x','giroscópio_Y','giroscópio_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
data = data[data.pessoa_x!=1629]

In [ ]:
data.pessoa_x.unique()

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
## number of events per user

plt.bar(data["pessoa_x"].value_counts().index, data["pessoa_x"].value_counts().values)
plt.show()

In [ ]:
#selecting only activities 0 to 3
#data = data[data["atividade"].isin([0,1,2,3,13,16])]

In [ ]:
# Define the mapping dictionary
#mapping_dict = {13:4,16:5}

In [ ]:
#data["atividade"] = data["atividade"].replace(to_replace=mapping_dict)

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
grouped_data = data.groupby(['pessoa_x'])['atividade'].value_counts().unstack()

In [ ]:
print(grouped_data)

In [ ]:
nan_values = grouped_data[grouped_data.isnull().any(axis=1)].index

In [ ]:
nan_values_array = nan_values.to_numpy()

In [ ]:
print(nan_values)

In [ ]:
data = data[~data['pessoa_x'].isin([1616, 1618, 1637, 1638, 1640, 1642])]

In [ ]:
grouped_data2 = data.groupby(['pessoa_x'])['atividade'].value_counts().unstack()

In [ ]:
print(grouped_data2)

# **Models**

In [ ]:
def delete_segments_not_fully_filled(df_data_windowed,df_label_windowed):
    indexes = np.where(df_label_windowed == -1)[0]
    unique_arr = np.unique(indexes, return_index=True)[0]
    new_df_label_windowed = np.delete(df_label_windowed,unique_arr,axis=0)
    new_df_data_windowed = np.delete(df_data_windowed,unique_arr,axis=0)
    return (new_df_data_windowed,new_df_label_windowed)

In [ ]:
def apply_segmentation_no_sliding_window(df_data):
# Define window size and step size
    window_size = 200
    step_size = 20

    # Calculate number of windows
    n_windows = int((df_data.shape[0] / window_size))

    # Create segmented data using sliding window
    segments = np.zeros((n_windows, window_size, df_data.shape[1]-1),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        #populate the 3d array only if the 200 segment has the value in label
        label_consistency = (df_data[['atividade']][start_idx:end_idx].nunique()==1) 
        if label_consistency[0] == True:
            segments[i] = df_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']][start_idx:end_idx]
            targets[i] = df_data[['atividade']][start_idx:end_idx]
    return (segments,targets)

In [ ]:
def apply_segmentation_with_sliding_window(df_data):
    #creating sliding window for train data
    import numpy as np

    window_size = 200
    step_size = 20

    n_windows = int((df_data.shape[0] - window_size) // step_size + 1)

    segments = np.zeros((n_windows, window_size, df_data.shape[1]-1),dtype=np.float32)
    targets = np.full((n_windows, window_size, 1),-1)
    for i in range(n_windows):
        start_idx = i * step_size
        end_idx = start_idx + window_size
        label_consistency = df_data[['atividade']][start_idx:end_idx].nunique()==1
        if label_consistency[0] == True:
            segments[i] = df_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']][start_idx:end_idx]
            targets[i] = df_data[['atividade']][start_idx:end_idx]
    return (segments,targets)

In [ ]:
print(data.pessoa_x.unique())

In [ ]:
X_test_data = data[data['pessoa_x'].isin([1600,1601, 1602, 1603, 1604])]

In [ ]:
print(X_test_data.shape)

In [ ]:
X_test_data = X_test_data.reset_index(drop = True)

In [ ]:
X_test_data.head()

In [ ]:
X_test_data.tail()

In [ ]:
X_test_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_test_data.shape[0]/data.shape[0]}")

In [ ]:
X_test_data['atividade'].value_counts()

In [ ]:
X_test_data = X_test_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_validation_data = data[~data.pessoa_x.isin([1600,1601, 1602, 1603, 1604])]

In [ ]:
print(X_train_validation_data.shape)

In [ ]:
X_train_validation_data = X_train_validation_data.reset_index(drop = True)

In [ ]:
X_train_validation_data.head()

In [ ]:
X_train_validation_data.tail()

In [ ]:
X_train_validation_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_train_validation_data.shape[0]/data.shape[0]}")

In [ ]:
X_train_validation_data['atividade'].value_counts()

In [ ]:
X_validation_data = X_train_validation_data[X_train_validation_data['pessoa_x'].isin([1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612,1613])]

In [ ]:
print(X_validation_data.shape)

In [ ]:
X_validation_data = X_validation_data.reset_index(drop = True)

In [ ]:
X_validation_data.head()

In [ ]:
X_validation_data.tail()

In [ ]:
X_validation_data.pessoa_x.unique()

In [ ]:
print(f"percentage of validation is {X_validation_data.shape[0]/X_train_validation_data.shape[0]}")

In [ ]:
X_validation_data['atividade'].value_counts()

In [ ]:
X_validation_data = X_validation_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_data = X_train_validation_data[~X_train_validation_data.pessoa_x.isin([1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612,1613])]

In [ ]:
X_train_data.pessoa_x.unique()

In [ ]:
X_train_data.shape

In [ ]:
X_train_data = X_train_data.reset_index(drop=True)

In [ ]:
print(f"percentage of validation is {X_train_data.shape[0]/X_train_validation_data.shape[0]}")

In [ ]:
X_train_data.head()

In [ ]:
X_train_data.tail()

In [ ]:
X_train_data = X_train_data[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
X_train_data.head()

In [ ]:
segments_test,targets_test = apply_segmentation_no_sliding_window(X_test_data)
print(targets_test.shape)
print(segments_test.shape)

In [ ]:
segments_test,targets_test = delete_segments_not_fully_filled(segments_test,targets_test)
print(targets_test.shape)
print(segments_test.shape)

In [ ]:
targets_test = targets_test.reshape(targets_test.shape[0],200)

In [ ]:
print(targets_test.shape)

In [ ]:
print(targets_test)

In [ ]:
labels_test = targets_test[:,0]

In [ ]:
print(labels_test)

In [ ]:
print(labels_test.shape)

In [ ]:
labels_test_categorical = to_categorical(labels_test,num_classes=6)

In [ ]:
print(labels_test_categorical)

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
segments_val,targets_val = apply_segmentation_with_sliding_window(X_validation_data)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
segments_val,targets_val = delete_segments_not_fully_filled(segments_val,targets_val)
print(targets_val.shape)
print(segments_val.shape)

In [ ]:
targets_val = targets_val.reshape(targets_val.shape[0],200)

In [ ]:
print(targets_val.shape)

In [ ]:
print(targets_val)

In [ ]:
labels_val = targets_val[:,0]

In [ ]:
print(labels_val)

In [ ]:
print(labels_val.shape)

In [ ]:
labels_val_categorical = to_categorical(labels_val,num_classes=6)

In [ ]:
print(labels_val_categorical)

In [ ]:
print(labels_val_categorical.shape)

In [ ]:
segments_train,targets_train = apply_segmentation_with_sliding_window(X_train_data)

In [ ]:
print(segments_train.shape)
print(targets_train.shape)

In [ ]:
segments_train, targets_train = delete_segments_not_fully_filled(segments_train,targets_train)

In [ ]:
print(segments_train.shape)
print(targets_train.shape)

In [ ]:
targets_train = targets_train.reshape(targets_train.shape[0],200)

In [ ]:
print(targets_train.shape)

In [ ]:
labels_train = targets_train[:,0]

In [ ]:
labels_train_categorical = to_categorical(labels_train,num_classes=6)

In [ ]:
print(labels_train_categorical)

In [ ]:
X_train = segments_train
y_train = labels_train_categorical
X_validation = segments_val
y_validation = labels_val_categorical

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [ ]:
''' X_train,y_train = shuffle_data(X_train,y_train)
X_validation,y_validation = shuffle_data(X_validation,y_validation) '''

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

In [ ]:
#from tensorflow.keras import regularizers

In [ ]:
''' def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))
  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation='relu'))
  model.add(tf.keras.layers.BatchNormalization())  
  model.add(tf.keras.layers.Dense(units=4, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model
 '''

In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(200,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(6, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model


In [ ]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [ ]:
tuner.search(x=X_train,
            y=y_train,
            epochs=50,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=32, use_multiprocessing=True)

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
print(segments_test)

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
print(labels_test_categorical)

In [ ]:
loss_test, accuracy_test = best_model.evaluate(segments_test, labels_test_categorical)

In [ ]:
predictions_test = best_model.predict(segments_test)

In [ ]:
print(predictions_test)

In [ ]:
classes_x_test=np.argmax(predictions_test,axis=1)

In [ ]:
print(classes_x_test)

In [ ]:
labels_test_argmax = np.argmax(labels_test_categorical,axis=1)

In [ ]:
print(labels_test_argmax)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm_test = confusion_matrix(labels_test_argmax, classes_x_test)

In [ ]:
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()

In [ ]:
print(classification_report(labels_test_argmax, classes_x_test))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)